# **R/S BENCHMARK — Neural Network DATASET GENERATION (from the PCE, not the emulator)**

This notebook loads the `pce_metamodel` files, one per time step.

## **1. Libraries**

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import dill
import numpy as np
import pandas as pd

from functions import *
from UQpy.distributions import Normal, JointIndependent

/home/casa-wand/Documentos/2024-1_victor_hugo_renata_maria/.venv/lib/python3.11/site-packages/UQpy/__init__.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## **2. Config**

Must match [`02_train_pce.ipynb`](02_train_pce.ipynb) — `r_mean`/`r_std`/`s_mean`/`s_std` rebuild
the same `joint`, `times` must be the same grid, and `n_latent_samples` names the `pce_metamodel`
files being loaded below (it plays no role in this notebook beyond that — no latent sampling
happens here).

`n_points` is new: how many fresh $(R, S)$ points to query per time step. Since a PCE evaluation is
cheap, this can be far denser than the design sample count used to fit the PCEs themselves.

In [2]:
r_mean = 5.0
r_std  = 0.8
s_mean = 2.0
s_std  = 0.6

n_latent_samples = 2500   # must match stage 1/2 — it names the pce_metamodel files
n_lambdas        = 4
n_points         = 5000   # (R, S) query points drawn per time step for the NN dataset

times = np.linspace(0, 150, 10, endpoint=True)  # must match stage 1/2
times

array([  0.        ,  16.66666667,  33.33333333,  50.        ,
        66.66666667,  83.33333333, 100.        , 116.66666667,
       133.33333333, 150.        ])

## 3. Rebuild the joint distribution

In [3]:
r_dist = Normal(loc=r_mean, scale=r_std)
s_dist = Normal(loc=s_mean, scale=s_std)
joint  = JointIndependent(marginals=[r_dist, s_dist])

## 4. Load the per-time-step PCE metamodels

One `pce_metamodel` per entry of `times`, as saved by `train_and_validate_pce_from_dataset_benchmark`
in stage 2.

In [4]:
pce_models = []
for t in times:
    with open(f'{n_latent_samples}_pce_metamodel_{t}_benchmark.pkl', 'rb') as f:
        pce_models.append(dill.load(f))

print(f"Loaded {len(pce_models)} PCE metamodels")

Loaded 10 PCE metamodels


## 5. Query the PCEs and stack the dataset

`generate_nn_dataset_benchmark` draws `n_points` fresh $(R, S)$ samples per time step, evaluates the
matching PCE on them, and stacks every time step into one dataframe with an explicit
`Time (years)` column.

In [5]:
print("="*60)
print("GENERATING THE NN DATASET FROM THE PCE MODELS")
print("="*60)

result = generate_nn_dataset_benchmark(
                                          pce_metamodels=pce_models,
                                          times=times,
                                          joint=joint,
                                          n_points=n_points,
                                          n_lambdas=n_lambdas,
                                          n_latent_samples=n_latent_samples,
                                          output_dir='.',
                                       )

df_nn = result['dataset_nn']
print(f"\nTotal rows: {len(df_nn)}")
df_nn.head()

GENERATING THE NN DATASET FROM THE PCE MODELS

----------------------------------------
GENERATING NN DATASET FROM 10 PCE MODELS
----------------------------------------
  t = 0.00 years: 5000 points queried from the PCE
  t = 16.67 years: 5000 points queried from the PCE
  t = 33.33 years: 5000 points queried from the PCE
  t = 50.00 years: 5000 points queried from the PCE
  t = 66.67 years: 5000 points queried from the PCE
  t = 83.33 years: 5000 points queried from the PCE
  t = 100.00 years: 5000 points queried from the PCE
  t = 116.67 years: 5000 points queried from the PCE
  t = 133.33 years: 5000 points queried from the PCE
  t = 150.00 years: 5000 points queried from the PCE
The NN dataset has been saved!

Total rows: 50000


,r,s,Time (years),lambda 1,lambda 2,lambda 3,lambda 4
0,5.110417,2.274454,0.0,2.837573,5.559482,0.143137,0.130569
1,3.435042,2.304231,0.0,1.133046,6.011376,0.138403,0.133853
2,5.195391,1.842210,0.0,3.354314,6.465245,0.144540,0.125109
3,4.566698,2.544968,0.0,2.023816,5.183452,0.142163,0.135642
4,5.815982,2.080980,0.0,3.736439,5.682511,0.143967,0.125393


## 6. Sanity check

In [6]:
df_nn[['r', 's', 'Time (years)', 'lambda 1', 'lambda 2', 'lambda 3', 'lambda 4']].describe()

,r,s,Time (years),lambda 1,lambda 2,lambda 3,lambda 4
count,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000
mean,4.999055,2.002467,75.000000,0.374165,7.860740,0.135472,0.129654
std,0.796860,0.600679,47.871834,1.838036,3.197509,0.010104,0.009067
min,1.563514,-0.561757,0.000000,-4.800225,-20.536834,0.070870,0.056796
25%,4.459958,1.594678,33.333333,-1.088307,6.036646,0.130141,0.125438
50%,4.996207,1.999220,75.000000,0.311368,7.056450,0.136553,0.130685
75%,5.535339,2.406403,116.666667,1.767125,8.739889,0.141936,0.135306
max,8.470426,4.620547,150.000000,6.515586,75.681276,0.291210,0.266094
